### الهدف من Notebook 2

نضيف عمود جديد اسمه **`label`** يحدد إذا الطلبية وصلت **متأخرة (Late)** أو **بموعدها (On-time)**.

### الخطوات

1. نقرأ الـ artifact اللي حفظناه من **Notebook 1**:
   `notebook1_ml_table.parquet`
   بدل ما نعيد شغل Notebook 1 من البداية.

2. نقارن بين:

   * `order_delivered_customer_date` → تاريخ التسليم الفعلي.
   * `order_estimated_delivery_date` → تاريخ التسليم المتوقع.

3. نبني الـ **label**:

   * إذا التسليم الفعلي > التسليم المتوقع → **Late**
   * غير هيك → **On-time**

4. نتحقق من الـ **label** على عينات من طلبيات حقيقية، عشان نتأكد إن المنطق شغال صح.

5. نفحص **توزيع الفئات** ونشوف نسبة:

   * Late
   * On-time

   عشان نعرف إذا فيه **Class Imbalance** ممكن يأثر لاحقًا على تدريب الموديل.

6. بالنهاية نحفظ الجدول الجديد كـ **artifact** لاستخدامه في المراحل الجاية.


### ليش بنقرأ الـ Artifact؟

لأننا ما بدنا نعيد شغل **Notebook 1** من الصفر. الفكرة هي **Artifact-Based Flow**:

* كل Notebook ببدأ من النتيجة اللي وصلها الـ Notebook اللي قبله.
* كل Notebook بكون **مستقل وقابل لإعادة التشغيل** لحاله.
* ما بنكرر نفس الكود والشغل أكثر من مرة.
* إذا عدّلنا أو غيرنا شيء في **Notebook 1**، بنعيد تشغيله، وبنحدّث الـ artifact، وبعدها الـ Notebooks اللي بعده بتستخدم النسخة الجديدة.

**يعني باختصار:** كل Notebook بستلم نتيجة اللي قبله وبيكمل عليها، بدل ما يعيد كل الشغل من البداية.


In [10]:
import pandas as pd

df = pd.read_parquet("data/processed/notebook1_ml_table.parquet")
print(df.shape)

(99441, 17)


In [12]:
"""ليش أضفنا فلتر order_status رغم إنه مش مذكور صراحة بخطوات المهمة؟

أضفنا فلتر استبعاد الطلبيات "الملغاة" (canceled) 
رغم إنه مش مكتوب صراحة بخطوات المهمة，
 لأنو المهمة نفسها بتطلب "التأكد إن الليبل صحيح قبل الاعتماد عليه".
   اكتشفنا 6 طلبيات ملغاة بس عندها تاريخ تسليم فعلي — تناقض منطقي بيخلي الليبل تبعها غير موثوق.
     فاستبعادها هو تطبيق فعلي لنفس المطلوب، مش إضافة خارجة عنه.
"""

before = len(df)
df = df[df["order_status"] != "canceled"]
after = len(df)

print(f"before: {before}")
print(f"after: {after}")
print(f"removed (canceled): {before - after}")

before: 99441
after: 98816
removed (canceled): 625


ليش الرقم صار 625 مش 6؟

الـ 6 طلبيات يلي اكتشفناها بـ Notebook 4 كانت بس الحالات النادرة يلي عندها تاريخ تسليم فعلي رغم إنها ملغاة (نجت من فلتر dropna). أما الـ 625 هلق فهي كل الطلبيات الملغاة بالجدول الأصلي (99,441 صف) — معظمها (619) أصلاً ما كان عندها تاريخ تسليم، وكانت رح تتحذف لاحقًا بالـ dropna على أي حال.

الفرق: الفلتر الجديد بيستبعد كل الطلبيات الملغاة دفعة وحدة، قبل الـ dropna — أشمل وأنضف من الاعتماد على dropna يشيل أغلبها بالصدفة. الرقم 625 صحيح ومنطقي، ومفيش تناقض مع الـ 6 يلي شفناها قبل.

In [13]:
"""
order_delivered_customer_date

هاد التاريخ الفعلي يلي الطلبية فعلًا وصلت فيه للزبون بإيده. يعني لما البائع/شركة الشحن سجلت "تم التسليم" فعليًا.

order_estimated_delivery_date

هاد التاريخ المتوقع يلي الموقع/الشركة وعدت الزبون إنو الطلبية رح توصله قبله (أو بيه). هاد التاريخ بينحسب وقت الشراء، قبل ما الطلبية حتى تنشحن.
"""

date_cols = ["order_delivered_customer_date", "order_estimated_delivery_date"]
df[date_cols].info()
df[date_cols].head()

<class 'pandas.DataFrame'>
Index: 98816 entries, 0 to 99440
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_delivered_customer_date  96470 non-null  datetime64[us]
 1   order_estimated_delivery_date  98816 non-null  datetime64[us]
dtypes: datetime64[us](2)
memory usage: 2.3 MB


,order_delivered_customer_date,order_estimated_delivery_date
0,2017-10-10 21:25:13,2017-10-18
1,2018-08-07 15:27:45,2018-08-13
2,2018-08-17 18:06:29,2018-09-04
3,2017-12-02 00:28:42,2017-12-15
4,2018-02-16 18:17:02,2018-02-26


- إجمالي الصفوف بعد فلتر canceled: 98,816 (بدل 99,441 الأصلي)
- order_delivered_customer_date: 96,470 غير فارغ → لسا فيه فجوة، بس صارت 
  أصغر (98,816 - 96,470 = 2,346 صف ناقص، بدل 2,965 بالنسخة القديمة)
- order_estimated_delivery_date: كامل (98,816 non-null) زي المتوقع
- الفرق بالفجوة (2,965 → 2,346) منطقي، لأنو جزء من الطلبيات الملغاة 
  يلي حذفناها كانت أصلاً من ضمن الطلبيات الناقصة التاريخ.

In [27]:
before = len(df)
print(f"Before dropping nulls: {before}")

Before dropping nulls: 96470


In [28]:
df = df.dropna(subset=["order_delivered_customer_date"])

In [29]:
after = len(df)
print(f"After dropping nulls: {after}")

After dropping nulls: 96470


In [26]:
print(f"before dropping : {before}")
print(f"after dropping : {after}")
print(f"removed: {before - after}")

before dropping : 96470
after dropping : 96470
removed: 0


الرقم "removed: 0" هون لأنو before وafter انحسبوا بعد التصفية 

المزدوجة (canceled + الفجوة الأصلية اتشالت فعليًا بخطوة سابقة/قراءة 

مختلفة) — القيمة الفعلية بعد كل الفلاتر: 96,470 صف نظيف جاهز لبناء الليبل.

In [30]:
import numpy as np

# np.where(condition, value_if_true, value_if_false)
df["label"] = np.where(
    df["order_delivered_customer_date"] > df["order_estimated_delivery_date"],
    "Late",
    "On-time",
)
df["label"].value_counts()

label
On-time    88644
Late        7826
Name: count, dtype: int64


مقارنة مع النسخة القديمة (قبل فلتر canceled):
On-time: 88,649 → 88,644 (فرق 5)
Late: 7,827 → 7,826 (فرق 1)

الفرق بسيط جدًا كما هو متوقع (استبعدنا حالات نادرة جدًا من الطلبيات 
الملغاة). النسبة العامة للـ imbalance (~92%/8%) ضلت شبه ثابتة.

### الأرقام (بعد التحديث)

* **On-time:** 88,644 (~91.9%)
* **Late:** 7,826 (~8.1%)

### المشكلة

فئة **On-time** أكبر بكثير من فئة **Late** — تقريبًا **11 طلبية On-time مقابل كل طلبية Late وحدة**.
هاد **Class Imbalance**، ونمط طبيعي بمشاكل زي التوصيل المتأخر، لأن الأغلبية بتوصل بالوقت والتأخير هو الاستثناء.

### ليش مهم ننتبهله؟

1. **بـ Split :**
   لازم نتأكد إنو نسبة `Late/On-time` تضل قريبة عبر `train/val/test`، وإلا التقييم بصير غير موثوق.

2. **بـ Notebook 6 (تدريب الموديل):**
   لو استخدمنا **Accuracy** كمقياس وحيد، الموديل ممكن "يغش" — لو قال `On-time` لكل الطلبيات بدون ما يتعلم شي، بيوصل Accuracy لـ ~92%!
   لازم نستخدم مقاييس أنسب مثل **F1-score, Precision/Recall, أو ROC-AUC**.

### الخلاصة

النسبة ضلت شبه ثابتة حتى بعد استبعاد الطلبيات الملغاة (فرق ضئيل جدًا: **5 صفوف On-time و1 صف Late بس**).

يعني فلتر `canceled` ما أثر على مشكلة الـ **imbalance** نفسها، وهاي الملاحظة لازم تضل حاضرة لغاية آخر خطوة بالمشروع (**تقييم الموديل**).


In [20]:
df[["order_delivered_customer_date", "order_estimated_delivery_date", "label"]].sample(20)

,order_delivered_customer_date,order_estimated_delivery_date,label
56894,2018-07-04 19:34:46,2018-07-16,On-time
16945,2018-07-26 16:41:56,2018-03-22,Late
70892,2017-05-09 11:04:10,2017-05-29,On-time
76828,2018-07-10 13:28:32,2018-07-24,On-time
16925,2018-08-02 19:48:37,2018-08-06,On-time
56120,2017-04-19 15:58:18,2017-04-28,On-time
81511,2018-01-09 19:59:20,2018-01-02,Late
24888,2018-06-30 18:43:26,2018-07-31,On-time
67927,2017-11-04 15:39:50,2017-11-08,On-time
97863,2018-04-20 00:06:29,2018-05-21,On-time


راجعنا عينة عشوائية من 20 صف للتأكد المنطق شغال صح بعد التحديث:
- صف 16945: التسليم 2018-07-26 بعد المتوقع 2018-03-22 → Late ✅ صحيح
- صف 81511: التسليم 2018-01-09 بعد المتوقع 2018-01-02 → Late ✅ صحيح
- باقي الصفوف: التسليم قبل أو بنفس المتوقع → On-time ✅ صحيح

المنطق شغال صحيح وسليم بعد إعادة التشغيل أيضًا.

In [33]:
same_day_late = df[
    (df["order_delivered_customer_date"].dt.date == df["order_estimated_delivery_date"].dt.date)
    & (df["label"] == "Late")
]
print("the number of same-day late orders is:", len(same_day_late))

the number of same-day late orders is: 1292


###

عمود `order_estimated_delivery_date` مسجل بدون وقت محدد، وبيتحسب تلقائيًا كمنتصف الليل `00:00:00`.

فأي طلبية توصل **بنفس يوم الموعد المتوقع** — بغض النظر عن الساعة — بتتصنف `Late` تلقائيًا، حتى لو المفروض تكون `On-time`.

### الحجم الفعلي

* **1,292 طلبية** متأثرة بهاي الظاهرة.
* من أصل **7,826 طلبية** مصنفة `Late`.
* يعني تقريبًا **16.5% من حالات Late** سببها مشكلة التوقيت، مش بالضرورة تأخير حقيقي.

### مثال

`2018-03-27 16:58:31` → وقت التسليم الفعلي
`2018-03-27 00:00:00` → الموعد المتوقع

بما إن `16:58 > 00:00`، انحسبت الطلبية **Late** رغم إنها وصلت بنفس اليوم.

### هل هاد خطأ؟

لأ، **مش خطأ بالكود**؛ هي نتيجة طبيعية لأن العمود المتوقع ما فيه وقت.

لكن بما إن الظاهرة بتأثر على **16.5% من حالات Late**، فهي نسبة كبيرة وبتستاهل ننتبهلها وما نتعامل معها كملاحظة بسيطة.




In [35]:
import os

os.makedirs("data/processed", exist_ok=True)

df.to_parquet("data/processed/notebook2_labeled_table.parquet", index=False)
print("Saved ✅")
print(df.shape)

Saved ✅
(96470, 18)
